## Data Exploration 

In [ ]:
import os
import glob
import numpy as np
import librosa
from IPython.display import Audio, display
from scipy.io import wavfile


root = "data"
digital_path = "EN_x/"
record_low_path = "EN_y1/"
record_high_path = "EN_y2"

digital_files = glob.glob(os.path.join(root, digital_path, "*.wav"))
record_low_files = glob.glob(os.path.join(root, record_low_path, "*.wav"))
record_high_files = glob.glob(os.path.join(root, record_high_path, "*.wav"))

print(len(digital_files), len(record_low_files), len(record_high_files))


In [ ]:
idx = 11
digital_file = digital_files[idx]
record_low_file = digital_file.replace("EN_x/","EN_y1/y1_")
record_high_file = digital_file.replace("EN_x/","EN_y2/y2_")
print(digital_file, record_low_file, record_high_file)
digital_rate, digital_data = wavfile.read(digital_file)
record_low_rate, record_low_data = wavfile.read(record_low_file)
record_high_rate, record_high_data = wavfile.read(record_high_file)

In [ ]:
print("Playing Digital Signal:")
display(Audio(data=digital_data, rate=digital_rate))

print("Playing Recorded Low-Quality Signal:")
display(Audio(data=record_low_data, rate=record_low_rate))

print("Playing Recorded High-Quality Signal:")
display(Audio(data=record_high_data, rate=record_high_rate))

In [ ]:
import matplotlib.pyplot as plt

digital_time = [i / digital_rate for i in range(len(digital_data))]
record_low_time = [i / record_low_rate for i in range(len(record_low_data))]
record_high_time = [i / record_high_rate for i in range(len(record_high_data))]


fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].plot(digital_time, digital_data, color='b')
axs[0].set_title('Digital Signal')
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Amplitude')

axs[1].plot(record_low_time, record_low_data, color='r')
axs[1].set_title('Recorded Low-Quality Signal')
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('Amplitude')

axs[2].plot(record_high_time, record_high_data, color='g')
axs[2].set_title('Recorded High-Quality Signal')
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import stft

duration_sec = 10

digital_samples = int(duration_sec * digital_rate)
record_low_samples = int(duration_sec * record_low_rate)
record_high_samples = int(duration_sec * record_high_rate)

digital_data = digital_data[:digital_samples]
record_low_data = record_low_data[:record_low_samples]
record_high_data = record_high_data[:record_high_samples]

f1, t1, Zxx1 = stft(digital_data, fs=digital_rate, nperseg=1024)
f2, t2, Zxx2 = stft(record_low_data, fs=record_low_rate, nperseg=1024)
f3, t3, Zxx3 = stft(record_high_data, fs=record_high_rate, nperseg=1024)


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6))

ax1.pcolormesh(t1, f1, np.abs(Zxx1), shading='gouraud')
ax1.set_title('Digital Signal Spectrogram (First 10 Seconds)')
ax1.set_xlabel('Time [sec]')
ax1.set_ylabel('Frequency [Hz]')
ax1.set_ylim([0, digital_rate / 2])  

ax2.pcolormesh(t2, f2, np.abs(Zxx2), shading='gouraud')
ax2.set_title('Recorded Low-Quality Signal Spectrogram (First 10 Seconds)')
ax2.set_xlabel('Time [sec]')
ax2.set_ylabel('Frequency [Hz]')
ax2.set_ylim([0, record_low_rate / 2])  

ax3.pcolormesh(t3, f3, np.abs(Zxx3), shading='gouraud')
ax3.set_title('Recorded High-Quality Signal Spectrogram (First 10 Seconds)')
ax3.set_xlabel('Time [sec]')
ax3.set_ylabel('Frequency [Hz]')
ax3.set_ylim([0, record_high_rate / 2])  

plt.tight_layout()
plt.show()



In [ ]:
import os
import librosa
from scipy.io import wavfile
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=5, stride_length=0.5, target_sampling_rate=16000):
    digital_waveforms = []
    record_low_waveforms = []

    for i in tqdm(range(30)):
        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"y2_{i}.wav")
        
        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate)
        
        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)
        
        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[start:start + segment_samples]
            
            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)
    
    return digital_waveforms, record_low_waveforms


digital_waveforms, record_high_waveforms = preprocess("data/EN_x", "data/EN_y2")

In [ ]:
print(len(digital_waveforms))

In [ ]:
import random

sample_index = random.randint(0, len(digital_waveforms) - 1)

digital_sample = digital_waveforms[sample_index]
record_high_sample = record_high_waveforms[sample_index]

plt.figure(figsize=(12, 5))

plt.subplot(2, 1, 1)
plt.plot(digital_sample)
plt.title("Digital Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.subplot(2, 1, 2)
plt.plot(record_high_sample)
plt.title("Record Low Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()


In [ ]:
print("Playing Digital Sample:")
display(Audio(digital_sample, rate=16000))

print("Playing Record Sample:")
display(Audio(record_high_sample, rate=16000))

## Data Loader

In [6]:
BATCH_SIZE = 2
NUM_WORKERS = 2
SHUFFLE = True
SAMPLE_RATE = 16000  
SEGMENT_LENGTH = 5
STRIDE_LENGTH = 0.5
STAGE = 1

In [7]:
import os
import librosa
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=SEGMENT_LENGTH, stride_length=STRIDE_LENGTH, target_sampling_rate=SAMPLE_RATE):
    total_files = 30
    train_end = int(0.8 * total_files)
    val_end = train_end + int(0.1 * total_files)

    train_digital_waveforms, val_digital_waveforms, test_digital_waveforms = [], [], []
    train_record_low_waveforms, val_record_low_waveforms, test_record_low_waveforms = [], [], []

    for i in tqdm(range(total_files)):
        if i < train_end:
            digital_waveforms = train_digital_waveforms
            record_low_waveforms = train_record_low_waveforms
        elif i < val_end:
            digital_waveforms = val_digital_waveforms
            record_low_waveforms = val_record_low_waveforms
        else:
            digital_waveforms = test_digital_waveforms
            record_low_waveforms = test_record_low_waveforms

        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"y{STAGE}_{i}.wav")

        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate)

        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)

        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[start:start + segment_samples]

            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)

    return (train_digital_waveforms, train_record_low_waveforms), \
           (val_digital_waveforms, val_record_low_waveforms), \
           (test_digital_waveforms, test_record_low_waveforms)


In [ ]:
(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess("data/EN_x", f"data/EN_y{STAGE}")


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class EqualizerDataset(Dataset):
    def __init__(self, digital_waveforms, record_low_waveforms, return_dict=False):
        assert len(digital_waveforms) == len(record_low_waveforms), \
            "Input and output waveforms lists must be of the same length"

        self.digital_waveforms = digital_waveforms
        self.record_low_waveforms = record_low_waveforms
        self.return_dict = return_dict

    def __len__(self):
        return len(self.digital_waveforms)

    def __getitem__(self, idx):
        digital_sample = torch.tensor(self.digital_waveforms[idx], dtype=torch.float32)
        record_low_sample = torch.tensor(self.record_low_waveforms[idx], dtype=torch.float32)
        if self.return_dict:
            return {'input_values': digital_sample, 'labels': record_low_sample}

        return digital_sample, record_low_sample

In [10]:
train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms)

In [ ]:
for digital_batch, record_low_batch in tqdm(train_dataset):
    print("Digital Batch Shape:", digital_batch.shape)
    print("Record Low Batch Shape:", record_low_batch.shape)
    break


## Model

### Wav2Vec Encoder

In [ ]:
import torch
from transformers import Wav2Vec2Model

wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

sample_waveform = torch.tensor(digital_batch).unsqueeze(0) 

print("Sample shape before forward pass:", sample_waveform.shape)

with torch.no_grad():
    wav2vec_output = wav2vec_model(sample_waveform)

print("Output shape from wav2vec 2.0:", wav2vec_output.last_hidden_state.shape)


In [10]:
import torch
import torch.nn as nn
from transformers import Wav2Vec2Model


class BLSTM(nn.Module):
    def __init__(self, dim, layers=2, bi=True):
        super().__init__()
        self.lstm = nn.LSTM(input_size=dim, hidden_size=dim, num_layers=layers, bidirectional=bi, batch_first=True)
        self.linear = nn.Linear(dim * 2, dim) if bi else None

    def forward(self, x):
        x, _ = self.lstm(x)  
        if self.linear:
            x = self.linear(x)
        return x  


class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.down_conv1 = nn.Conv1d(input_dim, hidden_dim, kernel_size=4, stride=2, padding=1)
        self.down_conv2 = nn.Conv1d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)
        self.down_conv3 = nn.Conv1d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)
        
        self.up_conv1 = nn.ConvTranspose1d(hidden_dim * 4, hidden_dim * 2, kernel_size=4, stride=2, padding=1)
        self.up_conv2 = nn.ConvTranspose1d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1)
        self.final_conv = nn.ConvTranspose1d(hidden_dim, 1, kernel_size=4, stride=2, padding=1)
        
    def forward(self, x):
        x1 = self.down_conv1(x) 
        x2 = self.down_conv2(x1)  
        x3 = self.down_conv3(x2)
        
        x = self.up_conv1(x3) 
        x = self.up_conv2(x) 
        x = self.final_conv(x)     
        
        return x


class Wav2VecEqualizer(nn.Module):
    def __init__(self, freeze_encoder=True, target_length=160000):
        super().__init__()
        
        self.encoder = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        if freeze_encoder:
            for name, param in self.encoder.named_parameters():
                if "pos_conv_embed" in name:
                    param.requires_grad = True
                else:
                    param.requires_grad = False

        # self.lstm = BLSTM(dim=768)
        
        self.decoder = Decoder(input_dim=499, hidden_dim=256)
        self.target_length = target_length
        self.fc = nn.Linear(768, target_length)

    def forward(self, x):
        x = self.encoder(x).last_hidden_state
        x = self.decoder(x).squeeze(1)        
        x = self.fc(x)
        return x

    

In [ ]:
model = Wav2VecEqualizer(freeze_encoder=True)
sample_input = torch.randn(2, 160000)  
output = model(sample_input)
print("Output shape:", output.shape)

### Demuc

In [ ]:
from IPython import display as disp
import torch
import torchaudio
from denoiser import pretrained
from denoiser.dsp import convert_audio
import librosa

model = pretrained.dns64().cpu()
wav, _ = librosa.load("data/EN_x/0.wav", sr=model.sample_rate)
with torch.no_grad():
    denoised = model(torch.FloatTensor(wav[None]))[0]
disp.display(disp.Audio(wav.data, rate=model.sample_rate))
disp.display(disp.Audio(denoised.data, rate=model.sample_rate))

## Trainer

In [31]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from loss import STFTLoss


stft_loss_fn = STFTLoss()

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions
    mse = ((preds - labels) ** 2).mean()
    stft_loss = stft_loss_fn(preds, labels)
    
    return {"mse": mse, "stft_loss": stft_loss}


class TrainerModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.mse_loss_fn = nn.MSELoss()  

    def forward(self, input_values, labels=None):
        outputs = self.model(input_values)
        loss = self.mse_loss_fn(outputs, labels) + stft_loss_fn(outputs, labels) if labels is not None else None
        
        return (loss, outputs) if loss is not None else outputs


In [ ]:
training_args = TrainingArguments(
    output_dir="assets",
    eval_strategy="epoch",
    learning_rate=5e-5,
    save_strategy='epoch',
    logging_steps=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="assets/logs",
    report_to="wandb",
    save_safetensors=False,
)

model = TrainerModelWrapper(Wav2VecEqualizer(freeze_encoder=False))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# trainer.train()

## Evaluation

### Stage 1

In [103]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs import DemucsEqualizer

STAGE = 1

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess("data/EN_x", f"data/EN_y{STAGE}")

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


100%|██████████| 30/30 [00:03<00:00,  9.67it/s]


In [104]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/model1/pytorch_model.bin"

model = DemucsEqualizer(device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset) - 1)
sample = train_dataset[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    enhanced_waveform = model(input_waveform).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=16000)) 

print("Target Audio:")
display(Audio(target_waveform, rate=16000))

print("Output Audio:")
display(Audio(enhanced_waveform, rate=16000))


/tmp/ipykernel_1043008/4143735128.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=device)


Original Digital Audio:


Target Audio:


Output Audio:


### Stage 2

In [105]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs import DemucsEqualizer, DoubleDemucsEqualizer

STAGE = 2

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess("data/EN_x", f"data/EN_y{STAGE}")

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


100%|██████████| 30/30 [00:03<00:00,  8.17it/s]


In [111]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/stage2/pytorch_model.bin"

model = DoubleDemucsEqualizer("assets/stage1/pytorch_model.bin", device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset) - 1)
sample = train_dataset[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    first_waveform = model.model2(input_waveform).squeeze(0)
    second_waveform = model.model1(first_waveform).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=16000)) 

print("Target Audio:")
display(Audio(target_waveform, rate=16000))

print("First Audio:")
display(Audio(first_waveform, rate=16000))

print("Second Audio:")
display(Audio(second_waveform, rate=16000))


/tmp/ipykernel_1043008/1708344447.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location=device)


Original Digital Audio:


Target Audio:


First Audio:


Second Audio:
